# GTA VI Comments - Exploratory Data Analysis

This notebook performs exploratory data analysis on the collected YouTube comments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from wordcloud import WordCloud

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load raw data
df_raw = pd.read_csv('../data/raw_comments.csv')
print(f"Raw comments: {len(df_raw):,}")

# Load cleaned data
df_clean = pd.read_csv('../data/clean_comments_validated.csv')
print(f"Clean comments: {len(df_clean):,}")

df_clean.head()

## 2. Basic Statistics

In [ ]:
print("Dataset Overview:")
print(f"Total comments: {len(df_clean):,}")
print(f"Date range: {df_clean['published_at'].min()} to {df_clean['published_at'].max()}")
print(f"\nText Statistics:")
print(f"Average length: {df_clean['cleaned_length'].mean():.1f} characters")
print(f"Average words: {df_clean['word_count'].mean():.1f} words")
print(f"\nEngagement:")
print(f"Total likes: {df_clean['like_count'].sum():,}")
print(f"Average likes: {df_clean['like_count'].mean():.2f}")

df_clean.describe()

## 3. Text Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Character count
axes[0].hist(df_clean['cleaned_length'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_title('Character Count Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')

# Word count
axes[1].hist(df_clean['word_count'], bins=50, color='lightcoral', edgecolor='black')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4. Language Distribution

In [ ]:
lang_counts = df_clean['language'].value_counts().head(10)

fig = px.bar(
    x=lang_counts.index,
    y=lang_counts.values,
    title='Top 10 Languages',
    labels={'x': 'Language', 'y': 'Count'},
    color=lang_counts.values,
    color_continuous_scale='viridis'
)
fig.show()

print(f"\nEnglish comments: {df_clean['is_english'].sum():,} ({100*df_clean['is_english'].mean():.1f}%)")

## 5. Toxicity Analysis

In [ ]:
tox_counts = df_clean['toxicity_level'].value_counts()

fig = px.pie(
    values=tox_counts.values,
    names=tox_counts.index,
    title='Toxicity Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

print("\nToxicity Breakdown:")
for level, count in tox_counts.items():
    print(f"{level}: {count:,} ({100*count/len(df_clean):.1f}%)")

## 6. Temporal Analysis

In [ ]:
df_time = df_clean.copy()
df_time['published_at'] = pd.to_datetime(df_time['published_at'])
df_time['date'] = df_time['published_at'].dt.date
df_time['hour'] = df_time['published_at'].dt.hour

# Comments over time
time_counts = df_time.groupby('date').size().reset_index(name='count')

fig = px.line(
    time_counts,
    x='date',
    y='count',
    title='Comments Over Time',
    labels={'date': 'Date', 'count': 'Number of Comments'}
)
fig.show()

# Comments by hour
hour_counts = df_time['hour'].value_counts().sort_index()

plt.figure(figsize=(12, 5))
plt.bar(hour_counts.index, hour_counts.values, color='teal')
plt.title('Comments by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Count')
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.3)
plt.show()

## 7. Engagement Analysis

In [ ]:
# Top liked comments
top_comments = df_clean.nlargest(10, 'like_count')[['text_clean', 'like_count']]
print("Top 10 Most Liked Comments:\n")
for idx, row in top_comments.iterrows():
    print(f"Likes: {row['like_count']}")
    print(f"Text: {row['text_clean'][:200]}...\n")

# Correlation between length and likes
correlation = df_clean['word_count'].corr(df_clean['like_count'])
print(f"\nCorrelation between word count and likes: {correlation:.3f}")

## 8. Word Cloud (English Comments Only)

In [ ]:
# Create word cloud from English comments
english_text = ' '.join(df_clean[df_clean['is_english']]['text_clean'].astype(str))

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    colormap='viridis',
    max_words=100
).generate(english_text)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Common Words in Comments', fontsize=20)
plt.tight_layout()
plt.show()

## 9. Quality Metrics

In [ ]:
print("Data Quality Metrics:\n")
print(f"High quality comments: {df_clean['is_high_quality'].sum():,} ({100*df_clean['is_high_quality'].mean():.1f}%)")
print(f"Spam comments: {df_clean['is_spam'].sum():,} ({100*df_clean['is_spam'].mean():.1f}%)")
print(f"English comments: {df_clean['is_english'].sum():,} ({100*df_clean['is_english'].mean():.1f}%)")
print(f"Safe comments: {(df_clean['toxicity_level'] == 'Safe').sum():,} ({100*(df_clean['toxicity_level'] == 'Safe').mean():.1f}%)")

## 10. Export Summary Statistics

In [ ]:
summary = {
    'total_comments': len(df_clean),
    'english_pct': 100 * df_clean['is_english'].mean(),
    'high_quality_pct': 100 * df_clean['is_high_quality'].mean(),
    'avg_length': df_clean['cleaned_length'].mean(),
    'avg_words': df_clean['word_count'].mean(),
    'total_likes': df_clean['like_count'].sum(),
}

print("\nSummary Statistics:")
for key, value in summary.items():
    print(f"{key}: {value:.2f}")